In [11]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch import nn, optim
from sklearn.metrics import classification_report
import numpy as np

In [12]:
class CustomImageDataset:
    def __init__(self, root_dir, transform = None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = []
        self.labels = []

        class_dirs = ['NORMAL', 'PNEUMONIA']
        for label, class_dir in enumerate(class_dirs):
            class_path = os.path.join(root_dir, class_dir)
            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)
                self.image_files.append(img_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [13]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [14]:
train_dataset = CustomImageDataset('/kaggle/input/labeled-chest-xray-images/chest_xray/train', transform=transform)
test_dataset = CustomImageDataset('/kaggle/input/labeled-chest-xray-images/chest_xray/test', transform = transform)

In [15]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

In [16]:
class ResNet101Classifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(ResNet101Classifier, self).__init__()
        # Загружаем предобученную ResNet-101
        self.resnet = models.resnet101(pretrained=pretrained)
        # Получаем количество входных признаков для последнего слоя
        in_features = self.resnet.fc.in_features
        # Заменяем последний слой на новый, подходящий под вашу задачу
        self.resnet.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

In [17]:
model = ResNet101Classifier(num_classes=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [18]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [19]:
num_epochs = 7
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f'Epochs {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}')

Epochs 1/7, Loss: 0.2311
Epochs 2/7, Loss: 0.0050
Epochs 3/7, Loss: 0.0020
Epochs 4/7, Loss: 0.3702
Epochs 5/7, Loss: 0.0159
Epochs 6/7, Loss: 0.0130
Epochs 7/7, Loss: 0.0062


In [20]:
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


print(classification_report(all_labels, all_preds, target_names=['NORMAL', 'PNEUMONIA']))

              precision    recall  f1-score   support

      NORMAL       0.96      0.76      0.85       234
   PNEUMONIA       0.87      0.98      0.92       390

    accuracy                           0.90       624
   macro avg       0.91      0.87      0.89       624
weighted avg       0.90      0.90      0.89       624

